In [ ]:
!pip install datasets pyarrow "selectolax<1.0" py-data-juicer
from huggingface_hub import login
login()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.9/240.9 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of fsspec[http] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10

In [3]:
import os
import pandas as pd
from datasets import load_dataset

print("Streaming CulturaX (sv) chunk...")
# The streaming=True flag prevents downloading the entire dataset
dataset = load_dataset("uonlp/CulturaX", "sv", split="train", streaming=True)

# Pull exactly 50,000 documents for the scaled test
raw_df = pd.DataFrame(dataset.take(50000))
raw_df["doc_id"] = raw_df.index.astype(str)
os.makedirs("culturax_sv_full_folder", exist_ok=True)

# Convert to Pandas and save as a partitioned Parquet file
#df = pd.DataFrame(chunk)
# Save as a compressed Parquet file inside the folder
raw_df.to_parquet("culturax_sv_full_folder/chunk_0.parquet", compression="snappy")
print(f"Saved {len(raw_df)} documents to the partitioned directory.")

Streaming CulturaX (sv) chunk...


Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Saved 50000 documents to the partitioned directory.


In [9]:
# %%writefile config.yaml
# # Point to the directory, not a single file, for parallel reading
# dataset_path: "culturax_sv_full_folder"
# export_path: "culturax_sv_cleaned"
# np: 2  # Colab provides 2 CPU cores by default

# process:
#   # 1. CHEAP FILTERS FIRST: Drop bad text before heavy processing
#   - text_length_filter:
#       min_len: 200
#       max_len: 8000

#   # 2. Strict Language ID: Discard machine-translated English or foreign noise
#   - language_id_score_filter:
#       lang: "sv"
#       min_score: 0.95

#   # 3. EXPENSIVE PARSING SECOND: Only clean HTML on verified Swedish paragraphs
#   - clean_html_mapper: {}

#   # 4. SCALABLE DEDUPLICATION: MinHash handles massive corpora without OOM errors
#   - document_minhash_deduplicator:
#       tokenization: space
#       num_permutations: 256
#       jaccard_threshold: 0.8

%%writefile config.yaml
dataset_path: "culturax_sv_full_folder"
export_path: "culturax_sv_cleaned.parquet"
np: 2

process:
  - text_length_filter:
      min_len: 200
      max_len: 8000

  - language_id_score_filter:
      lang: "sv"
      min_score: 0.95

  - clean_html_mapper: {}
  - clean_links_mapper: {}
  - fix_unicode_mapper: {}

  - document_simhash_deduplicator:
      tokenization: space

Overwriting config.yaml


In [10]:
!dj-process --config config.yaml

import pandas as pd

cleaned_df = pd.read_parquet("culturax_sv_cleaned.parquet")
print(f"Original documents: 50,000")
print(f"Cleaned seed documents: {len(cleaned_df)}")
display(cleaned_df.head(3))

2026-10-05 17:37:56.413 | INFO     | data_juicer.config.config:1020 - dataset_path config is set and a valid local path
2026-10-05 17:37:56.597 | INFO     | data_juicer.config.config:920 - 🔍 Checking for job resumption: 20261005_173756_ccd153
2026-10-05 17:37:56.598 | WARNING  | data_juicer.config.config:1426 - No config file found in work directory: /content/20261005_173756_ccd153
2026-10-05 17:37:56.599 | INFO     | data_juicer.config.config:1667 - Back up the input config file [/content/config.yaml] to [/content/20261005_173756_ccd153/config.yaml]
2026-10-05 17:37:56.600 | INFO     | data_juicer.config.config:1408 - 💾 Saved CLI arguments to: /content/20261005_173756_ccd153/cli.yaml
2026-10-05 17:37:56.609 | INFO     | data_juicer.config.config:1692 - Configuration table: 
╒═════════════════════════════════════╤══════════════════════════════════════════════════════════════════════════════════════╕
│ key                                 │ values                                         

,text,timestamp,url,source,doc_id
0,Med ett Minor Field Studies stipendium kan du ...,2020/04/07 07:43:23,https://www.du.se/sv/Utbildning/under-studiern...,mC4,0
1,"Kreativ marknadskommunikation, Kurs, - Luleå t...",2021/12/07 03:36:07,https://www.ltu.se/edu/course/M00/M0035N/M0035...,mC4,1
2,Den bästa idén genom tiderna | Aftonbladet\nDe...,2017/10/21 13:35:26,https://www.aftonbladet.se/nyheter/article1163...,mC4,2


In [14]:
import pandas as pd
import re

# 1. Load both datasets
raw_df = pd.read_parquet("culturax_sv_full_folder/chunk_0.parquet")
clean_df = pd.read_parquet("culturax_sv_cleaned.parquet")

# 2. Dataset-level Summary
raw_lengths = raw_df["text"].str.len()
clean_lengths = clean_df["text"].str.len()

print("=" * 60)
print(f"Dataset Size Reduction: {len(raw_df):,} -> {len(clean_df):,} "
      f"({(1 - len(clean_df)/len(raw_df))*100:.1f}% dropped)")
print(f"Original Text Length: min={raw_lengths.min()}, median={raw_lengths.median():.0f}, max={raw_lengths.max()}")
print(f"Cleaned Text Length:  min={clean_lengths.min()}, median={clean_lengths.median():.0f}, max={clean_lengths.max()}")
print("=" * 60)

# 3. Sanity Checks on Cleaned Data
html_tags_remaining = clean_df["text"].apply(lambda t: bool(re.search(r"<[^>]+>", t))).sum()
exact_duplicates = clean_df["text"].duplicated().sum()
under_min_len = (clean_lengths < 200).sum()
over_max_len = (clean_lengths > 8000).sum()

print("Integrity Checks:")
print(f" - Documents with remaining HTML tags: {html_tags_remaining}")
print(f" - Exact duplicate documents:          {exact_duplicates}")
print(f" - Documents outside length bounds:    {under_min_len + over_max_len}")
print("=" * 60)

# 4. Print 3 Cleaned Text Samples (Untruncated preview)
print("\nSample Cleaned Documents:\n")
for i, sample in enumerate(clean_df["text"].sample(3, random_state=42), 1):
    preview = sample[:400].replace("\n", " ") + ("..." if len(sample) > 400 else "")
    print(f"[{i}] ({len(sample)} chars):")
    print(f"{preview}\n")

Dataset Size Reduction: 50,000 -> 43,886 (12.2% dropped)
Original Text Length: min=103, median=1824, max=163188
Cleaned Text Length:  min=92, median=1738, max=7995
Integrity Checks:
 - Documents with remaining HTML tags: 74
 - Exact duplicate documents:          0
 - Documents outside length bounds:    5

Sample Cleaned Documents:

[1] (374 chars):
Anabola Steroider är det farligt, Öka energi,, Fördelarna med tillsatsen: för Methandrostenolone – steroiderna – officiellt namn. och även infertilitet. betyder det att produkten är helt ogynnsamma biverkningar, lämplig för en idrottsman kan du påskynda återhämtning efter träning., vinst som kan etablera gradvis. inte att gå och chanserna. till 25 vitaminer och väsentliga

[2] (495 chars):
Huvudregeln är att du ska betala ett avtal som du har ingått förutsatt att ett bolag eller kreditgivaren har presterat sin del av avtalet. Men det finns också regler som säger att ett betalningskrav kan bli för gammalt vilket gör att du inte behöver betala

In [18]:
import pandas as pd
import difflib

# 1. Merge raw and clean dataframes on doc_id
merged = pd.merge(
    raw_df[["doc_id", "text", "url"]].rename(columns={"text": "raw_text"}),
    clean_df[["doc_id", "text"]].rename(columns={"text": "clean_text"}),
    on="doc_id"
)

# 2. Filter exclusively for documents that were transformed in-place
modified_docs = merged[merged["raw_text"] != merged["clean_text"]]

print(f"Surviving documents: {len(merged):,}")
print(f"Documents modified in-place: {len(modified_docs):,}\n")

if not modified_docs.empty:
    sample = modified_docs.sample(1, random_state=42).iloc[0]

    print(f"Document ID: {sample['doc_id']}")
    #print(f"Source URL:  {sample['url']}\n")
    print(f"Source URL:  {sample['url']}\n")

    print("=== RAW (BEFORE) ===")
    print(sample["raw_text"][:1200] + "...\n")

    print("=== CLEANED (AFTER) ===")
    print(sample["clean_text"][:1200] + "...\n")

    # 3. Show the exact character-level differences
    print("=== REMOVED ARTIFACTS / DIFF ===")
    diff = difflib.ndiff(sample["raw_text"][:300].splitlines(keepends=True),
                         sample["clean_text"][:300].splitlines(keepends=True))
    for line in diff:
        if line.startswith("- ") or line.startswith("+ "):
            print(line.strip())
else:
    print("No modified documents found. All survivors passed through filters without inline edits.")

Surviving documents: 43,886
Documents modified in-place: 6,212

Document ID: 35602
Source URL:  

=== RAW (BEFORE) ===
Resultatet av vår undersökning är oroväckande. Med syfte att kunna skicka stora mängder, ofta skräddarsydd, reklam samlar många nätbutiker in och lagrar långt mer uppgifter om sina kunder än nödvändigt. Dessutom vidareförmedlas ofta uppgifterna till andra företag, inom och utanför EU.
I vissa fall får konsumenten inte ens en chans att acceptera företagets personuppgiftspolicy, utan man förutsätts acceptera den bara genom att vara inne på företagets webbsida.
Därför anmäler vi nu Ticnet, Adlibris, IKEA, CDON och Booking.com till datainspektionen för brott mot personuppgiftslagen.
På den lilla lappen i våra kläder står ofta en rad olika material. Men vad är egentligen polyester? Hur tillverkas hampa och vilka material är minst dåligt för miljön? Nedan listar vi de flesta material du kan stöta på i dina kläder och berättar mer om hu ...
Svetskommissionen arbetar med ett s